### genim 라이브러리 모델에서 GridSearchCV를 이용하는 방법
- sklearn에서 제공하는 GridSearchCV는 모델에서 fit(), transform(), predict()과 같은 함수들을 이용하여 최적의 매개변수의 값을 찾는 객체
- gensim 라이브러리에는 해당 함수들이 존재하지 않는다.
- Word2Vec과 같은 객체을 생성하여 변수에 저장하고 fit(), tranform() 함수를 생성하여 하나로 묶어두기 위해서 새로운 Class을 선언
- 해당 Class를 이용하여 Pipeline, Kfold, GridSearchCV를 이용

In [1]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from konlpy.tag import Komoran
import numpy as np 
import pandas as pd 
from gensim.models import Word2Vec
# sklearn에서 기본적으로 제공해주는 함수들을 상속 받기 위해서 특정 객체를 로드 
from sklearn.base import BaseEstimator, TransformerMixin

In [2]:
# 토큰화 함수 생성 
komoran = Komoran()
tokenizer = lambda x : [ word for word in komoran.morphs(x) ]
tokenizer("나는 학교에 간다")

['나', '는', '학교', '에', '간다']

In [3]:
# Class 선언 : Word2Vec를 GridSearchCV에서 사용하기 위해서 선언
class Word2VecVectorizer( BaseEstimator, TransformerMixin ):
    # BaseEstimator : get_params(), set_params()와 같은 함수의 기능을 상속 받는다
    # TransformerMixin :  fit()와 tranform() 함수만 선언하면 fit_transform() 함수를 이용 가능

    # 생성자 함수 -> class가 생성이 될때 기본적으로 안에서 사용할 변수들을 지정 (데이터 대입)
    # Word2Vec에서 사용할 인자 값들을 생성자 함수에서 미리 받아온다.
    def __init__(
            self,               # 자기 자신 : 객체가 생성된 위치
            tokenizer = None,   # 토큰화 함수 (기본값은 None)
            vector_size = 100,  # 벡터화 된 데이터의 차원의 수 지정 
            min_count = 5,      # 전체 문서에서 최소 등장 횟수 지정
            window = 5,         # 중심 단어와 주변 단어들의 거리 제한
            sg = 1,             # 단어 예측 방식( 0 : CBOW, 1 : skip-gram )
            epochs = 100,       # 반복 학습의 횟수를 지정
            workers = 1,        # 계산에 사용할 스레드의 수 
            seed = 42
    ):
        self.tokenizer = tokenizer
        self.vector_size = vector_size
        self.min_count = min_count
        self.window = window
        self.sg = sg
        self.epochs = epochs
        self.workers = workers
        self.seed = seed

        # 모델이랑 단어 사전을 저장할 빈 공간을 생성
        # 일반적인 문법 : class를 선언 시 self.변수(객체 변수)들은 생성자 함수에서 생성한다
        self.model = None
        self.voca = None
    # 총 4개의 메서드를 생성 : 토큰화 , 학습, 문장 데이터를 평균 단위 벡터로 생성하는 함수, 변형

    # 토큰화 메서드 
    def to_token(self, sentences):
        # sentences : 문장들의 목록
        # 만약에 토큰화 함수가 존재하지 않는다면 self.tokenizer가 None인 경우 -> split() 함수를 이용
        if self.tokenizer is None:
            result = []
            for sentence in sentences:
                token = list(sentence.split())
                result.append(token)
            # result = [ [word for word in sentence.split()] for sentence in sentences ]
        else:
            # result = []
            # for sentence in sentences:
            #     token = self.tokenizer(sentence)
            #     result.append(token)
            result = [self.tokenizer(sentence) for sentence in sentences] 
        return result

    # 학습 메서드 : fit() 함수 생성 
    # sklearn안에 모델들의 fit() 함수에서 인자 값들은? 독립변수, 종속변수
    def fit(self, X, y):
        # X : 독립 변수 (문장 목록)
        # y : 종속 변수 (1차원 데이터)
        # sentences는 토큰화된 문장 데이터 
        sentences = self.to_token(X)
        # print(sentences)
        # Word2Vec에 학습 
        self.model = Word2Vec(
            sentences= sentences, 
            vector_size= self.vector_size, 
            window = self.window, 
            min_count = self.min_count, 
            sg  = self.sg, 
            epochs = self.epochs, 
            workers = self.workers,
            seed = self.seed
        )
        # 학습 모델이 생성이 되었으니 단어 사전에 데이터를 입력 
        self.voca = self.model.wv.key_to_index.keys()
        return self
    
    # 문장 벡터 생성하는 함수 
    def doc_vec(self, token):
        # token : 토큰화 된 문장 데이터 (1개의 문장)
        vectors = []
        for word in token:
            if word in self.model.wv.index_to_key:
                vec = self.model.wv[word]
                vectors.append(vec)
        # vectors 데이터가 존재하지 않는 경우 -> 특정 문장에서 단어들이 단어사전에 존재하지 않을때
        if vectors:
            result = np.mean(vectors, axis=0)
        else:
            result = np.zeros(self.model.vector_size)
        return result
    
    # 변형 함수 생성 
    # sklearn 안에 transform() 함수와 같이 인자 값 구성 
    def transform(self, X):
        # 토큰화
        sentences = self.to_token(X)
        # 벡터화(임베딩)
        result = []
        for token in sentences:
            vec = self.doc_vec(token)
            result.append(vec)
        return np.array(result)
                

In [4]:
docs = [
    '상품의 품질이 아주 좋다', 
    '배송이 너무 느리다'
]
label = [1, 0]

In [5]:
vec_class = Word2VecVectorizer(tokenizer, min_count=1)

In [6]:
# tokens = vec_class.to_token(docs)

In [7]:
vec_class.fit(docs, label)

,tokenizer,<function <la...002CA2D2F6200>
,vector_size,100
,min_count,1
,window,5
,sg,1
,epochs,100
,workers,1
,seed,42


In [8]:
# vec_class.doc_vec(tokens[0])

In [9]:
vec_class.transform(docs)

array([[-3.90404835e-03,  1.19129498e-03,  9.29581758e-04,
         2.35296180e-03, -1.47208071e-03,  3.83133418e-03,
        -2.15800735e-03, -1.96607551e-03, -2.96800863e-04,
        -6.95949711e-04,  1.79377303e-03,  3.14609963e-03,
         2.60756467e-03,  2.32866616e-03,  4.86230571e-03,
         2.78585218e-03, -1.07222621e-03, -3.41052795e-03,
        -2.65562488e-03,  1.29136461e-04, -3.26449191e-03,
        -7.95223925e-04, -1.94079673e-03,  1.27059850e-03,
         2.01618229e-03, -3.22334585e-03,  5.38224820e-04,
        -1.74110779e-03,  2.33143917e-03, -4.92594845e-04,
         8.26582022e-04, -3.63157853e-03, -1.37886941e-03,
        -1.15463568e-03, -7.82374642e-04,  1.54310639e-03,
         1.91604288e-03, -4.56508668e-03, -7.29850202e-04,
         3.62180616e-03,  1.58831943e-03,  2.60791439e-03,
         3.24082305e-03, -3.89193697e-03, -5.29001642e-04,
        -4.38771630e-03, -7.55552377e-04, -1.08175515e-03,
         4.08019125e-03, -1.05616637e-04,  3.19783343e-0

In [10]:
vec_class.fit_transform(docs, label)[0][0]

np.float32(-0.0039040484)

In [11]:
vec_class.get_params()

{'epochs': 100,
 'min_count': 1,
 'seed': 42,
 'sg': 1,
 'tokenizer': <function __main__.<lambda>(x)>,
 'vector_size': 100,
 'window': 5,
 'workers': 1}

In [12]:
vec_class.set_params(
    workers = 2, 
    sg = 0
)

,tokenizer,<function <la...002CA2D2F6200>
,vector_size,100
,min_count,1
,window,5
,sg,0
,epochs,100
,workers,2
,seed,42


In [13]:

vec_class.fit_transform(docs, label)[0][0]

np.float32(-0.0038800766)

In [14]:
df = pd.read_csv("../data/ratings_train.txt", sep='\t')

In [15]:
df.dropna(inplace=True)
df.drop_duplicates('document', inplace=True)

df2 = df[:200]

df2['label'].value_counts()

label
1    101
0     99
Name: count, dtype: int64

In [16]:

# 파이프 라인  생성 
pipe = Pipeline(
    [
        ('emb', Word2VecVectorizer(tokenizer=tokenizer, min_count=1)), 
        ('logi', LogisticRegression())
    ]
)

In [17]:
# pipe을 이용하여 fit() 함수 호출 
# emb에서 fit_tranform()함수를 호출하고 logi에 학습 
pipe.fit(df2['document'].values, df2['label'].values)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('emb', ...), ('logi', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,tokenizer,<function <la...002CA2D2F6200>
,vector_size,100
,min_count,1
,window,5
,sg,1
,epochs,100
,workers,1


In [18]:
df3 = df.tail(100)
X_test = df3['document'].values
y_test = df3['label'].values

In [19]:
pred = pipe.predict(X_test)

In [20]:
print(classification_report(pred, y_test))

              precision    recall  f1-score   support

           0       0.51      0.60      0.55        40
           1       0.70      0.62      0.65        60

    accuracy                           0.61       100
   macro avg       0.60      0.61      0.60       100
weighted avg       0.62      0.61      0.61       100



In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [22]:
# 파라미터 조합 
grid_params = {
    'emb__vector_size' : [100, 200], 
    'emb__sg' : [0, 1], 
    'logi__C' : [0.8, 1.0]
}

In [23]:
grid = GridSearchCV(
    estimator= pipe, 
    param_grid= grid_params, 
    cv = cv, 
    verbose=1
)

In [24]:
X = df2['document'].values
y = df2['label'].values

In [25]:
grid.fit(X, y)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'emb__sg': [0, 1], 'emb__vector_size': [100, 200], 'logi__C': [0.8, 1.0]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is disp

In [26]:
print("GridSearch 기준 최적의 파라미터 : ", grid.best_params_)
print('GridSearch 최적의 점수 : ', grid.best_score_)

GridSearch 기준 최적의 파라미터 :  {'emb__sg': 1, 'emb__vector_size': 100, 'logi__C': 1.0}
GridSearch 최적의 점수 :  0.64


In [27]:
pred = grid.predict(X_test)

print(classification_report(pred, y_test))

              precision    recall  f1-score   support

           0       0.51      0.60      0.55        40
           1       0.70      0.62      0.65        60

    accuracy                           0.61       100
   macro avg       0.60      0.61      0.60       100
weighted avg       0.62      0.61      0.61       100

